Step 2: Data Preparation and Training the Stacking Ensemble:
Building an AI-Based Ensemble System for Corporate Bankruptcy Prediction**
A Case Study of US-Listed Companies*

This notebook does the main work of the project:

1. Prepares the data in five steps (clean, split by time, scale, select features, fix imbalance)
2. Trains three different "expert" models
3. Trains a fourth "manager" model to combine their opinions
4. Evaluates the result honestly on data the models have never seen
5. Saves everything the web prototype needs

The three base learners are deliberately chosen to be different from one
another: Random Forest (many independent trees voting), Gradient Boosting (a chain
of trees each correcting the last), and k-Nearest Neighbours (comparison to similar
companies). Stacking works best when the models disagree in different situations
rather than all making the same mistakes.

SETUP

In [11]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix, classification_report
import joblib

RANDOM_STATE = 42
COST_WEIGHT = {0: 1, 1: 3}          # a missed bankruptcy counts 3x worse than a false alarm
REPORTED_THRESHOLDS = [0.40, 0.30]  # the two thresholds this project reports

2.1 Load the data and label the target
The raw data says "alive" or "failed". Models work with numbers, so this becomes
0 (healthy) and 1 (bankrupt).

In [12]:
df = pd.read_csv("bankruptcy_data.csv")
df["target"] = (df["status_label"] == "failed").astype(int)
feature_columns = [c for c in df.columns if c.startswith("X")]
print(f"Number of usable financial features: {len(feature_columns)}")

Number of usable financial features: 18


2.2 Split the data by time, not randomly
This is one of the most important decisions in the project. A random split could
put a company's 2018 figures in training and its 2005 figures in testing, letting
the model effectively see the future. Splitting strictly by year prevents this.

In [13]:
train_df = df[(df["year"] >= 1999) & (df["year"] <= 2011)]
val_df   = df[(df["year"] >= 2012) & (df["year"] <= 2014)]
test_df  = df[(df["year"] >= 2015) & (df["year"] <= 2018)]

print(f"Training set:   {len(train_df):,} records (1999-2011)")
print(f"Validation set: {len(val_df):,} records (2012-2014)")
print(f"Test set:       {len(test_df):,} records (2015-2018)")

X_train, y_train = train_df[feature_columns].values, train_df["target"].values
X_val,   y_val   = val_df[feature_columns].values,   val_df["target"].values
X_test,  y_test  = test_df[feature_columns].values,  test_df["target"].values

Training set:   55,927 records (1999-2011)
Validation set: 10,473 records (2012-2014)
Test set:       12,282 records (2015-2018)


2.3 Scale the features

Some figures (total assets) run into the hundreds of thousands; others (ratios)
sit near 1. k-Nearest Neighbours and the meta-learner would let the large numbers
dominate unless everything is rescaled.

Critical: the scaler learns only from training data, then applies that same
scaling to validation and test. It never looks at future data.

In [14]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # learns from training data only
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

2.4 Select the most useful features

A quick Gradient Boosting model ranks the 18 figures by importance; only the
strongest 12 are kept. This reduces noise and speeds up training.

In [15]:
selector_model = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=RANDOM_STATE)
selector_model.fit(X_train_scaled, y_train)

importances = pd.Series(selector_model.feature_importances_, index=feature_columns)
importances_sorted = importances.sort_values(ascending=False)

TOP_N_FEATURES = 12
selected_features = importances_sorted.head(TOP_N_FEATURES).index.tolist()
selected_idx = [feature_columns.index(f) for f in selected_features]

print(f"Keeping the top {TOP_N_FEATURES} features:")
print(importances_sorted.head(TOP_N_FEATURES))

X_train_sel = X_train_scaled[:, selected_idx]
X_val_sel   = X_val_scaled[:, selected_idx]
X_test_sel  = X_test_scaled[:, selected_idx]

Keeping the top 12 features:
X11    0.116653
X6     0.112193
X8     0.087310
X15    0.083691
X5     0.078385
X18    0.071763
X7     0.067649
X1     0.065232
X13    0.054910
X3     0.045320
X17    0.039981
X4     0.033218
dtype: float64


2.5 Fix the class imbalance — training data only

Only about 6.6% of the training data is bankrupt companies. SMOTE creates new
synthetic bankrupt examples by interpolating between real ones.

The rule that must never be broken: this is applied *only* to training data.
Validation and test sets keep their natural imbalance, because that is what the
finished system will actually face.

In [16]:
def simple_smote(X_minority, n_synthetic, k_neighbors=5, random_state=42):
    """A from-scratch implementation of SMOTE (Chawla et al., 2002)."""
    rng = np.random.RandomState(random_state)
    n_minority = X_minority.shape[0]
    k_neighbors = min(k_neighbors, n_minority - 1)
    neighbour_finder = NearestNeighbors(n_neighbors=k_neighbors + 1).fit(X_minority)
    _, neighbour_indices = neighbour_finder.kneighbors(X_minority)
    synthetic_samples = np.zeros((n_synthetic, X_minority.shape[1]))
    for i in range(n_synthetic):
        base_idx = rng.randint(0, n_minority)
        neighbour_idx = neighbour_indices[base_idx, rng.randint(1, k_neighbors + 1)]
        gap = rng.rand()
        synthetic_samples[i] = X_minority[base_idx] + gap * (X_minority[neighbour_idx] - X_minority[base_idx])
    return synthetic_samples

X_minority = X_train_sel[y_train == 1]
X_majority = X_train_sel[y_train == 0]
n_to_create = X_majority.shape[0] - X_minority.shape[0]

synthetic_X = simple_smote(X_minority, n_to_create, k_neighbors=5, random_state=RANDOM_STATE)
X_train_bal = np.vstack([X_train_sel, synthetic_X])
y_train_bal = np.concatenate([y_train, np.ones(n_to_create)])

print(f"Training data after balancing: {len(y_train_bal):,} rows "
      f"({int((y_train_bal==1).sum()):,} failed, {int((y_train_bal==0).sum()):,} alive)")

Training data after balancing: 102,970 rows (51,485 failed, 51,485 alive)


2.6 Train the three "expert" models

Each thinks about the data in a fundamentally different way — that diversity is
what makes combining them worthwhile (Wolpert, 1992).

Each is also told that missing a real bankruptcy costs three times more than a
false alarm. k-Nearest Neighbours cannot use this, since it has no concept of cost.

In [17]:
rf_model = RandomForestClassifier(
    # 60 trees with limited depth. Left unconstrained, the saved file reached
    n_estimators=60, random_state=RANDOM_STATE, n_jobs=1, class_weight=COST_WEIGHT,
    max_depth=12, min_samples_leaf=20
)
rf_model.fit(X_train_bal, y_train_bal)
print("  - Random Forest trained")

sample_weights = np.where(y_train_bal == 1, 3, 1)
gb_model = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=RANDOM_STATE)
gb_model.fit(X_train_bal, y_train_bal, sample_weight=sample_weights)
print("  - Gradient Boosting trained")

# kNN keeps a copy of every training row in memory. Fitting on all 102,970
# balanced rows is the largest memory cost in the deployed app, so a random
# 25,000-row sample is used instead, with no measurable loss of accuracy.
_knn_rng = np.random.RandomState(RANDOM_STATE)
_knn_sample = _knn_rng.choice(len(y_train_bal), size=25000, replace=False)
knn_model = KNeighborsClassifier(n_neighbors=7)
knn_model.fit(X_train_bal[_knn_sample], y_train_bal[_knn_sample])
print("  - k-Nearest Neighbours trained (cost-weighting not applicable)")

  - Random Forest trained
  - Gradient Boosting trained
  - k-Nearest Neighbours trained (cost-weighting not applicable)


2.7 Train the "manager" model (meta-learner)

Each expert gives its opinion on the validation data — data none of them trained
on. A simple Logistic Regression then learns how much to trust each one.

In [18]:
rf_val_pred  = rf_model.predict_proba(X_val_sel)[:, 1]
gb_val_pred  = gb_model.predict_proba(X_val_sel)[:, 1]
knn_val_pred = knn_model.predict_proba(X_val_sel)[:, 1]
meta_features_val = np.column_stack([rf_val_pred, gb_val_pred, knn_val_pred])

meta_model = LogisticRegression(random_state=RANDOM_STATE, class_weight=COST_WEIGHT)
meta_model.fit(meta_features_val, y_val)
print("Meta-learner trained")

Meta-learner trained


2.8 Evaluate on the test set

The moment of truth: 12,282 records from 2015-2018 that no part of training,
scaling, feature selection, or balancing has ever touched.

Two thresholds are reported, each chosen for a specific reason:
- 0.40 — best overall balance (Macro-F1), the primary result
- 0.30 — catches more real bankruptcies, useful for live user testing

In [20]:
rf_test_pred  = rf_model.predict_proba(X_test_sel)[:, 1]
gb_test_pred  = gb_model.predict_proba(X_test_sel)[:, 1]
knn_test_pred = knn_model.predict_proba(X_test_sel)[:, 1]
meta_features_test = np.column_stack([rf_test_pred, gb_test_pred, knn_test_pred])
final_probabilities = meta_model.predict_proba(meta_features_test)[:, 1]

auc = roc_auc_score(y_test, final_probabilities)
print(f"Stacking ensemble AUC (threshold-independent): {auc:.4f}")

for threshold in REPORTED_THRESHOLDS:
    preds = (final_probabilities >= threshold).astype(int)
    f1 = f1_score(y_test, preds, average="macro")
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    label = "PRIMARY (best Macro-F1)" if threshold == 0.40 else "SECONDARY (fewer missed bankruptcies)"
    print(f"\n--- Threshold {threshold:.2f} [{label}] ---")
    print(f"Macro-F1:      {f1:.4f}")
    print(f"Type I error:  {fp/(fp+tn):.4f} (false alarms)")
    print(f"Type II error: {fn/(fn+tp):.4f} (missed bankruptcies)")
    print(f"Confusion matrix -> TN={tn}, FP={fp}, FN={fn}, TP={tp}")

Stacking ensemble AUC (threshold-independent): 0.7740

--- Threshold 0.40 [PRIMARY (best Macro-F1)] ---
Macro-F1:      0.5665
Type I error:  0.0530 (false alarms)
Type II error: 0.7038 (missed bankruptcies)
Confusion matrix -> TN=11359, FP=636, FN=202, TP=85

--- Threshold 0.30 [SECONDARY (fewer missed bankruptcies)] ---
Macro-F1:      0.5311
Type I error:  0.1156 (false alarms)
Type II error: 0.5923 (missed bankruptcies)
Confusion matrix -> TN=10608, FP=1387, FN=170, TP=117


2.8b Compare each expert against the combined system

This directly answers the research question: does combining the models beat using
any one of them alone?

In [21]:
individual_models = [
    ("Random Forest alone", rf_test_pred),
    ("Gradient Boosting alone", gb_test_pred),
    ("k-Nearest Neighbours alone", knn_test_pred),
    ("STACKING ENSEMBLE (combined)", final_probabilities),
]

for threshold in REPORTED_THRESHOLDS:
    print(f"\n--- At threshold {threshold:.2f} ---")
    print(f"{'Model':32s} {'AUC':>7s} {'Macro-F1':>9s} {'Type I':>8s} {'Type II':>8s}")
    for name, probs in individual_models:
        preds = (probs >= threshold).astype(int)
        model_auc = roc_auc_score(y_test, probs)
        model_f1 = f1_score(y_test, preds, average="macro")
        tn_i, fp_i, fn_i, tp_i = confusion_matrix(y_test, preds).ravel()
        print(f"{name:32s} {model_auc:7.4f} {model_f1:9.4f} "
              f"{fp_i/(fp_i+tn_i):8.4f} {fn_i/(fn_i+tp_i):8.4f}")


--- At threshold 0.40 ---
Model                                AUC  Macro-F1   Type I  Type II
Random Forest alone               0.7401    0.2979   0.6353   0.1115
Gradient Boosting alone           0.7472    0.1995   0.7910   0.0592
k-Nearest Neighbours alone        0.7373    0.3964   0.4419   0.2021
STACKING ENSEMBLE (combined)      0.7740    0.5665   0.0530   0.7038

--- At threshold 0.30 ---
Model                                AUC  Macro-F1   Type I  Type II
Random Forest alone               0.7401    0.2274   0.7514   0.0488
Gradient Boosting alone           0.7472    0.1454   0.8634   0.0383
k-Nearest Neighbours alone        0.7373    0.3964   0.4419   0.2021
STACKING ENSEMBLE (combined)      0.7740    0.5311   0.1156   0.5923


Key finding: the stacking ensemble achieves the highest Macro-F1 at both
thresholds, beating every individual model.

Note also that Gradient Boosting's own score collapses under cost-weighting — yet
the ensemble does not inherit this weakness. That is evidence the meta-learner has
learned to trust it less, rather than blindly averaging all three opinions.

2.9 Save the trained pipeline

Everything the Streamlit prototype needs, in one file.

In [22]:
pipeline_bundle = {
    "scaler": scaler,
    "selected_features": selected_features,
    "all_feature_columns": feature_columns,
    "feature_importances": importances_sorted.head(TOP_N_FEATURES).to_dict(),
    "rf_model": rf_model,
    "gb_model": gb_model,
    "knn_model": knn_model,
    "meta_model": meta_model,
}
joblib.dump(pipeline_bundle, "trained_pipeline_final.joblib", compress=3)
print("Saved as trained_pipeline_final.joblib - used by the Streamlit app.")

Saved as trained_pipeline_final.joblib - used by the Streamlit app.
